# LlamaIndex for Complex Document QA

Build sophisticated document intelligence systems — multi-document Q&A, sub-question decomposition, and knowledge graph retrieval.

**Topics:** LlamaIndex nodes/indexes, VectorStoreIndex, sub-question decomposition, RouterQueryEngine, knowledge graphs

## 1. LlamaIndex Core Concepts — Nodes, Indexes, Query Engines

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
import os

# Global settings (avoids passing llm/embed everywhere)
Settings.llm = OpenAI(model='gpt-4o-mini', api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small', api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
Settings.chunk_size = 512
Settings.chunk_overlap = 50

# Node: atomic unit of information (chunk + metadata + relationships)
def create_linked_nodes(texts: list[str], source: str) -> list[TextNode]:
    """Create nodes with explicit previous/next relationships."""
    nodes = [TextNode(text=t, metadata={'source': source, 'chunk_index': i}) for i, t in enumerate(texts)]
    
    for i in range(len(nodes)):
        if i > 0:
            nodes[i].relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(node_id=nodes[i-1].node_id)
        if i < len(nodes) - 1:
            nodes[i].relationships[NodeRelationship.NEXT] = RelatedNodeInfo(node_id=nodes[i+1].node_id)
    
    return nodes

# LlamaIndex architecture
architecture = {
    'Documents': 'Raw text + metadata (PDF, HTML, markdown, etc.)',
    'Nodes': 'Chunked documents with relationships and metadata',
    'Index': 'Data structure for efficient retrieval (vector, tree, keyword)',
    'Retriever': 'Fetches relevant nodes given a query',
    'Response Synthesizer': 'Synthesizes answer from retrieved nodes + LLM',
    'Query Engine': 'End-to-end: retriever + synthesizer + post-processing',
}

print('LlamaIndex pipeline architecture:')
for component, role in architecture.items():
    print(f'  {component:<25}: {role}')
print()
print('Key difference from LangChain:')
print('  LlamaIndex: optimized for document intelligence (complex Q&A, hierarchical docs)')
print('  LangChain:  optimized for chains and agents (tool use, memory, workflows)')

## 2. VectorStoreIndex and SimpleDirectoryReader

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# Load documents from multiple formats
def load_and_index(data_dir: str) -> VectorStoreIndex:
    """Load docs from directory and build a vector index."""
    documents = SimpleDirectoryReader(
        input_dir=data_dir,
        recursive=True,
        required_exts=['.pdf', '.txt', '.md', '.docx'],
        filename_as_id=True,  # use filename as document ID
    ).load_data()
    
    print(f'Loaded {len(documents)} documents')
    
    # Parse into nodes with sentence splitter
    parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
    nodes = parser.get_nodes_from_documents(documents)
    print(f'Parsed into {len(nodes)} nodes')
    
    # Build index with persistent ChromaDB backend
    chroma_client = chromadb.PersistentClient(path='./llama_chroma')
    chroma_collection = chroma_client.get_or_create_collection('knowledge_base')
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    return index

def configure_query_engine(index: VectorStoreIndex, top_k: int = 5):
    return index.as_query_engine(
        similarity_top_k=top_k,
        response_mode='tree_summarize',  # recursively summarizes chunks
        streaming=True,
    )

# Response modes
modes = [
    ('refine', 'Iteratively refine answer with each chunk (best quality)'),
    ('compact', 'Compact chunks into fewer prompts then refine (faster)'),
    ('tree_summarize', 'Build summary tree bottom-up (best for long docs)'),
    ('simple_summarize', 'Single LLM call with all chunks (fastest, lowest quality)'),
    ('no_text', 'Return only retrieved nodes (use your own synthesis)'),
]
print('Query engine response modes:')
for mode, desc in modes:
    print(f'  {mode:<20}: {desc}')

## 3. Sub-Question Query Decomposition

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata

def build_multi_doc_agent(
    indexes: dict[str, VectorStoreIndex],
) -> SubQuestionQueryEngine:
    """Answer complex questions spanning multiple documents.
    
    SubQuestionQueryEngine:
    1. Decomposes complex question into sub-questions
    2. Routes each sub-question to the right document/tool
    3. Combines sub-answers into a final answer
    """
    tools = [
        QueryEngineTool(
            query_engine=idx.as_query_engine(),
            metadata=ToolMetadata(name=name, description=f'Answers questions about {name}'),
        )
        for name, idx in indexes.items()
    ]
    
    return SubQuestionQueryEngine.from_defaults(
        query_engine_tools=tools,
        use_async=True,   # answer sub-questions in parallel!
    )

# Example decomposition trace
complex_question = 'Compare the attention mechanisms in GPT-3 and BERT, and explain which is better for classification tasks.'

decomposition_trace = [
    ('Sub-Q 1', 'gpt3_paper', 'How does the attention mechanism work in GPT-3?'),
    ('Sub-Q 2', 'bert_paper', 'How does the attention mechanism work in BERT?'),
    ('Sub-Q 3', 'bert_paper', 'How does BERT perform on classification tasks?'),
    ('Synthesis', 'llm', 'Combine all answers to answer the original question.'),
]

print(f'Complex question: "{complex_question}"')
print()
print('SubQuestionQueryEngine decomposition:')
for sub_q, source, question in decomposition_trace:
    print(f'  [{sub_q}] → {source}: "{question}"')
print()
print('Sub-questions 1-3 run in parallel → saves 2x latency!')

## 4. RouterQueryEngine — Route to Right Index

In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector, PydanticMultiSelector
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core import SummaryIndex, KeywordTableIndex

def build_router_engine(index: VectorStoreIndex) -> RouterQueryEngine:
    """Route queries to specialized engines based on query type."""
    
    # Three specialized engines for different query types
    vector_tool = QueryEngineTool.from_defaults(
        query_engine=index.as_query_engine(similarity_top_k=5),
        description='Useful for semantic questions about concepts, definitions, and explanations.',
    )
    
    summary_tool = QueryEngineTool.from_defaults(
        query_engine=SummaryIndex.from_documents([]).as_query_engine(),
        description='Useful for summarizing entire documents or getting an overview.',
    )
    
    keyword_tool = QueryEngineTool.from_defaults(
        query_engine=KeywordTableIndex.from_documents([]).as_query_engine(),
        description='Useful for finding documents that contain specific keywords or technical terms.',
    )
    
    return RouterQueryEngine(
        selector=LLMSingleSelector.from_defaults(),  # LLM picks the best engine
        query_engine_tools=[vector_tool, summary_tool, keyword_tool],
        verbose=True,
    )

# Routing examples
routing_examples = [
    ('What is the difference between BERT and GPT?', 'vector_engine', 'semantic comparison'),
    ('Summarize the entire ML textbook.', 'summary_engine', 'document overview'),
    ('Find all documents mentioning PagedAttention.', 'keyword_engine', 'exact term search'),
    ('What are the main topics covered?', 'summary_engine', 'high-level overview'),
    ('How does dropout prevent overfitting?', 'vector_engine', 'concept explanation'),
]

print('RouterQueryEngine — automatic query routing:')
print(f'{"Query":<45} {"→ Engine":<18} {"Reason"}')
print('-' * 80)
for query, engine, reason in routing_examples:
    print(f'{query[:44]:<45} {engine:<18} {reason}')

## 5. Knowledge Graph Index

In [ ]:
from llama_index.core import KnowledgeGraphIndex, StorageContext
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core.schema import TextNode

def build_knowledge_graph(documents: list) -> KnowledgeGraphIndex:
    """Extract a knowledge graph from documents — great for structured domains.
    
    KG Index extracts triplets: (subject, predicate, object)
    e.g., (BERT, is_a, transformer) (BERT, trained_on, Wikipedia)
    """
    graph_store = SimpleGraphStore()
    storage_context = StorageContext.from_defaults(graph_store=graph_store)
    
    kg_index = KnowledgeGraphIndex.from_documents(
        documents,
        max_triplets_per_chunk=5,  # triplets per text chunk
        storage_context=storage_context,
        include_embeddings=True,   # hybrid: KG + vector search
    )
    return kg_index

# Simulated extracted triplets from ML literature
extracted_triplets = [
    ('BERT', 'is_a', 'transformer model'),
    ('BERT', 'uses', 'bidirectional attention'),
    ('BERT', 'trained_on', 'Wikipedia and BookCorpus'),
    ('BERT', 'achieves_sota_on', 'GLUE benchmark'),
    ('GPT', 'uses', 'causal (unidirectional) attention'),
    ('GPT', 'is_better_for', 'text generation'),
    ('BERT', 'is_better_for', 'text classification'),
    ('attention mechanism', 'was_introduced_in', 'Attention Is All You Need (2017)'),
    ('transformer', 'replaced', 'RNNs for NLP'),
]

print('Knowledge graph triplets extracted from ML literature:')
print(f'{"Subject":<25} {"Predicate":<25} {"Object"}')
print('-' * 70)
for subj, pred, obj in extracted_triplets:
    print(f'{subj:<25} {pred:<25} {obj}')
print()
print('KG query advantage: "What does BERT use?" → graph traversal → bidirectional attention')
print('vs vector search which might retrieve unrelated chunks about attention.')

## Exercises

1. **Multi-document comparison:** Build a SubQuestionQueryEngine over 3 ML papers. Ask it to compare the methodologies, results, and limitations of all three. Measure response quality vs a single-document query engine.

2. **Hybrid KG + vector:** Combine KnowledgeGraphIndex with VectorStoreIndex using a custom retriever that queries both, deduplicates results, and merges them before synthesis.

3. **Auto-updating index:** Implement a file watcher that monitors a directory for new/changed documents and automatically re-indexes them in LlamaIndex, using document IDs to update only changed nodes.